# Course 9 — Support Vector Machines

From the geometry of separating hyperplanes to the kernel trick, multi-class extensions,
and the connection to logistic regression.

**Sections**
1. Hyperplanes & the Maximum-Margin Classifier
2. Soft Margin — the Support Vector Classifier
3. Feature Expansion & the Kernel Trick
4. ROC Curves, Multi-class SVMs & p ≫ n (Khan)
5. SVM vs Logistic Regression
6. Recap
7. Exercises

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'pyproject.toml').exists() and p != p.parent:
    p = p.parent
if str(p) not in sys.path:
    sys.path.insert(0, str(p))
from shared.data_utils import load_dataset

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.datasets import make_blobs, make_moons, make_circles, load_iris
from sklearn.metrics import (
    accuracy_score, confusion_matrix, roc_curve, auc, classification_report
)
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

---
## 1. Hyperplanes & the Maximum-Margin Classifier

### What is a hyperplane?

In *p* dimensions a **hyperplane** is a flat affine subspace of dimension *p − 1*:

$$\beta_0 + \beta_1 X_1 + \beta_2 X_2 + \cdots + \beta_p X_p = 0$$

- In 2-D this is a line; in 3-D it is a plane.
- The vector **β** = (β₁, …, βₚ) is the **normal vector** — perpendicular to the surface.
- Points with f(X) > 0 lie on one side; f(X) < 0 on the other.
- The signed value |f(x)| / ‖β‖ equals the perpendicular distance from x to the hyperplane.

In [ ]:
# Visualise a hyperplane (line) in 2-D with its normal vector
# Hyperplane: 0.8*X1 + 0.6*X2 - 6 = 0  →  same as the ISLR slide
beta0, beta1, beta2 = -6, 0.8, 0.6

x1 = np.linspace(-2, 10, 200)
x2_line = -(beta0 + beta1 * x1) / beta2          # the hyperplane
x2_up   = x2_line + 1.6 / beta2                  # parallel shift +1.6
x2_dn   = x2_line - 4.0 / beta2                  # parallel shift -4.0

# Sample points on each parallel plane
pts_blue  = np.array([[6, 6.5], [7, 5], [8, 4.2], [6, 3.8]])   # f>0 side
pts_green = np.array([[2, 7.2], [2, 6.2], [4, 0.4], [5, 0.3]]) # f<0 side

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(x1, x2_line, 'b-',  lw=2, label=r'$\beta_0+\beta_1 X_1+\beta_2 X_2=0$')
ax.plot(x1, x2_up,   'b--', lw=1.2, alpha=0.6, label='shifted +1.6')
ax.plot(x1, x2_dn,   'g--', lw=1.2, alpha=0.6, label='shifted −4.0')

# Normal vector arrow at midpoint
mid_x, mid_y = 5, -(beta0 + beta1*5)/beta2
ax.annotate('', xy=(mid_x + beta1*1.8, mid_y + beta2*1.8),
            xytext=(mid_x, mid_y),
            arrowprops=dict(arrowstyle='->', color='red', lw=2))
ax.text(mid_x + beta1*2.0, mid_y + beta2*2.0, r'$\beta=(\beta_1,\beta_2)$',
        color='red', fontsize=10)

ax.scatter(*pts_blue.T,  s=60, color='steelblue', zorder=5)
ax.scatter(*pts_green.T, s=60, color='green',     zorder=5)
ax.set(xlim=(-2, 10), ylim=(-2, 10), xlabel='X₁', ylabel='X₂',
       title='Hyperplane in 2-D  (β₁=0.8, β₂=0.6, β₀=−6)')
ax.legend(fontsize=9); plt.tight_layout(); plt.show()

# Compute signed distances for two example points
for label, pt in [('blue (6,6.5)', [6, 6.5]), ('green (2,7.2)', [2, 7.2])]:
    fx = beta0 + beta1*pt[0] + beta2*pt[1]
    dist = fx / np.linalg.norm([beta1, beta2])
    print(f'{label}: f(x)={fx:.2f}   signed distance={dist:.2f}')

### Separating hyperplanes

Code the two classes as $y_i = +1$ (blue) and $y_i = -1$ (mauve).
A hyperplane *separates* the classes when:

$$y_i \cdot f(x_i) > 0 \quad \text{for all } i$$

When a separating hyperplane exists there are **infinitely many** of them —
any small rotation or translation still separates. The **Maximal Margin Classifier (MMC)**
picks the unique one that maximises the gap (margin) to both classes.

The MMC is brittle: a single outlier near the boundary can dramatically shift the
hyperplane, giving a completely different margin. This fragility motivates the
**soft-margin extension**, which allows a small number of violations in exchange
for a more stable, wider boundary.

In [ ]:
# Show infinitely many separating hyperplanes vs the unique max-margin one
np.random.seed(3)
X_sep = np.r_[np.random.randn(15, 2) + [2, 2],
               np.random.randn(15, 2) - [1, 1]]
y_sep = np.array([1]*15 + [-1]*15)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: three valid separating lines
ax = axes[0]
ax.scatter(X_sep[y_sep==1,0],  X_sep[y_sep==1,1],  s=40, label='y=+1')
ax.scatter(X_sep[y_sep==-1,0], X_sep[y_sep==-1,1], s=40, color='C3', label='y=−1')
xx = np.linspace(-4, 5, 200)
for slope, intercept in [(-1, 1.5), (-0.5, 0.8), (-1.5, 0.6)]:
    ax.plot(xx, slope*xx + intercept, 'k-', lw=1.2, alpha=0.5)
ax.set(title='Infinitely many separating hyperplanes', xlim=(-4,5), ylim=(-4,5))
ax.legend()

# Right: the max-margin one
ax = axes[1]
svc_hard = SVC(kernel='linear', C=1e6).fit(X_sep, y_sep)
w, b = svc_hard.coef_[0], svc_hard.intercept_[0]
yy_mm = -(w[0]*xx + b) / w[1]
margin_mm = 1 / np.linalg.norm(w)
slope_mm = -w[0]/w[1]
ax.scatter(X_sep[y_sep==1,0],  X_sep[y_sep==1,1],  s=40)
ax.scatter(X_sep[y_sep==-1,0], X_sep[y_sep==-1,1], s=40, color='C3')
ax.plot(xx, yy_mm, 'k-', lw=2)
ax.plot(xx, yy_mm + margin_mm*np.sqrt(1+slope_mm**2), 'k--', alpha=0.5)
ax.plot(xx, yy_mm - margin_mm*np.sqrt(1+slope_mm**2), 'k--', alpha=0.5)
ax.scatter(*svc_hard.support_vectors_.T, s=120, facecolors='none',
           edgecolors='k', linewidths=1.5, label='support vectors')
ax.set(title=f'Maximum-margin classifier  (margin={2*margin_mm:.2f})',
       xlim=(-4,5), ylim=(-4,5))
ax.legend()
plt.tight_layout(); plt.show()

### The MMC optimization problem

Among all separating hyperplanes, find the one that maximises M (the half-width of the margin):

$$\text{maximize}_{\beta_0,\boldsymbol{\beta}}\; M \quad \text{subject to} \quad \|\boldsymbol{\beta}\|=1,\quad y_i(\beta_0 + \boldsymbol{\beta}^\top x_i) \geq M \;\forall i$$

Equivalently: **minimise ½‖β‖²** subject to $y_i(\beta_0 + \boldsymbol{\beta}^\top x_i) \geq 1$.

This is a *convex quadratic program* with a unique global solution. The points
lying on the margin boundaries are the **support vectors** — all other points
can move without changing the solution.

The dual formulation reveals why: the solution depends only on training points whose
KKT multipliers $\hat{\alpha}_i > 0$ — exactly the support vectors on the margin
boundary. All other points have $\hat{\alpha}_i = 0$ and are irrelevant to the
decision boundary.

In [ ]:
# Fit the MMC on clean separable data and annotate support vectors
X_blob, y_blob = make_blobs(n_samples=80, centers=2, cluster_std=0.9, random_state=1)
svc_mmc = SVC(kernel='linear', C=1e6).fit(X_blob, y_blob)

w = svc_mmc.coef_[0];  b = svc_mmc.intercept_[0]
xx = np.linspace(X_blob[:,0].min()-1, X_blob[:,0].max()+1, 300)
yy = -(w[0]*xx + b) / w[1]
margin = 1 / np.linalg.norm(w)
slope  = -w[0] / w[1]
yy_up  = yy + margin * np.sqrt(1 + slope**2)
yy_dn  = yy - margin * np.sqrt(1 + slope**2)

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(X_blob[y_blob==0,0], X_blob[y_blob==0,1], s=35, label='class 0')
ax.scatter(X_blob[y_blob==1,0], X_blob[y_blob==1,1], s=35, color='C3', label='class 1')
ax.plot(xx, yy,    'k-',  lw=1.8, label='decision boundary')
ax.plot(xx, yy_up, 'k--', alpha=0.5, label=f'margin (width≈{2*margin:.2f})')
ax.plot(xx, yy_dn, 'k--', alpha=0.5)
# only support vectors determine the boundary — the rest of the training data is irrelevant
ax.scatter(*svc_mmc.support_vectors_.T, s=130, facecolors='none',
           edgecolors='black', linewidths=1.8, label='support vectors')
ax.legend(fontsize=9); ax.set_title('Maximum-Margin Classifier')
plt.tight_layout(); plt.show()
print(f'Support vectors per class: {svc_mmc.n_support_}')
print(f'Total support vectors    : {len(svc_mmc.support_vectors_)}')
print(f'Margin width             : {2*margin:.4f}')

---
## 2. Soft Margin — the Support Vector Classifier

Real data is rarely perfectly separable. Two problems arise:

1. **Non-separable** — no hyperplane separates the classes at all.
2. **Noisy but separable** — a hard margin fits the outlier perfectly but produces
   a very narrow, high-variance boundary.

The **Support Vector Classifier** (soft-margin SVM) introduces slack variables εᵢ ≥ 0
that allow some points to violate the margin:

$$\text{maximize}\; M \quad \text{subject to}\quad \|\beta\|=1,\quad y_i(\beta_0+\beta^\top x_i) \geq M(1-\varepsilon_i),\quad \varepsilon_i\geq 0,\quad \sum_i \varepsilon_i \leq C$$

- εᵢ = 0: point is on the correct side of the margin.
- 0 < εᵢ ≤ 1: inside the margin but correctly classified.
- εᵢ > 1: misclassified.

**C** is the total budget for slack — it is a **regularisation parameter**:
- **Small C** (sklearn): tight budget → narrow margin → few SVs → overfits.
- **Large C** (sklearn): loose budget → wide margin → many SVs → underfits.

*Note*: sklearn's `C` is the inverse of ISLR's λ. Larger sklearn C → less regularisation.

In [ ]:
# 4-panel: varying C on overlapping 2-class data (matches ISLR Fig 9.7)
X_ov, y_ov = make_blobs(n_samples=100, centers=2, cluster_std=1.5, random_state=5)
# C controls the margin width: small C = wide margin (more violations allowed), large C = narrow margin (few violations)
C_vals = [0.01, 0.1, 1.0, 100.0]

fig, axes = plt.subplots(2, 2, figsize=(11, 8))
for ax, C in zip(axes.flat, C_vals):
    svc = SVC(kernel='linear', C=C).fit(X_ov, y_ov)
    w = svc.coef_[0]; b = svc.intercept_[0]
    xx = np.linspace(X_ov[:,0].min()-1, X_ov[:,0].max()+1, 300)
    yy = -(w[0]*xx + b) / w[1]
    margin = 1 / np.linalg.norm(w)
    slope  = -w[0]/w[1]
    ax.scatter(X_ov[y_ov==0,0], X_ov[y_ov==0,1], s=20, alpha=0.7)
    ax.scatter(X_ov[y_ov==1,0], X_ov[y_ov==1,1], s=20, alpha=0.7, color='C3')
    ax.plot(xx, yy, 'k-', lw=1.8)
    ax.plot(xx, yy + margin*np.sqrt(1+slope**2), 'k--', alpha=0.4)
    ax.plot(xx, yy - margin*np.sqrt(1+slope**2), 'k--', alpha=0.4)
    # only support vectors determine the boundary — the rest of the training data is irrelevant
    ax.scatter(*svc.support_vectors_.T, s=90, facecolors='none',
               edgecolors='k', linewidths=1.3)
    ax.set_title(f'C={C}  |  margin≈{2*margin:.2f}  |  #SVs={len(svc.support_vectors_)}',
                 fontsize=10)
plt.suptitle('Support Vector Classifier: effect of C', fontsize=13)
plt.tight_layout(); plt.show()

### Bias-variance tradeoff via C

A small C forces a wide margin, accepting many violations — higher bias, lower variance.
A large C insists on few violations — lower bias on the training set but higher variance
(sensitive to individual noisy points). The optimal C is found by cross-validation.

C is the SVM's main regularisation knob: small C = high bias (wide margin, many support
vectors), large C = high variance (tight margin, few support vectors, sensitive to noise).
Unlike ridge or lasso, this single scalar controls both the geometric margin and the
tolerance for misclassification.

In [ ]:
# Train vs CV accuracy as C varies — linear vs RBF kernel
X_tr, X_te, y_tr, y_te = train_test_split(X_ov, y_ov, test_size=0.3, random_state=0)
C_grid = np.logspace(-3, 3, 30)

results_kv = {}
for kernel, kw in [('linear', {}), ('rbf', {'gamma': 'scale'})]:
    train_acc, cv_acc = [], []
    for C in C_grid:
        svc = SVC(kernel=kernel, C=C, **kw)
        train_acc.append(svc.fit(X_tr, y_tr).score(X_tr, y_tr))
        cv_acc.append(cross_val_score(svc, X_tr, y_tr, cv=5).mean())
    results_kv[kernel] = (train_acc, cv_acc)

fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
for ax, (kernel, (train_acc, cv_acc)) in zip(axes, results_kv.items()):
    ax.semilogx(C_grid, train_acc, lw=2,       label='Train accuracy')
    ax.semilogx(C_grid, cv_acc,   lw=2, ls='--', label='5-fold CV accuracy')
    best_C = C_grid[np.argmax(cv_acc)]
    ax.axvline(best_C, color='grey', lw=1, ls=':', alpha=0.8,
               label=f'best C={best_C:.3f}')
    ax.set(xlabel='C (log scale)', ylabel='Accuracy',
           title=f'SVC accuracy vs C  ({kernel} kernel)')
    ax.legend(); ax.grid(alpha=0.3)
plt.suptitle('Bias–variance tradeoff via C: linear vs RBF', fontsize=12)
plt.tight_layout(); plt.show()

for kernel, (train_acc, cv_acc) in results_kv.items():
    best_C = C_grid[np.argmax(cv_acc)]
    test_acc = SVC(kernel=kernel, C=best_C,
                   **({'gamma': 'scale'} if kernel == 'rbf' else {})
                  ).fit(X_tr, y_tr).score(X_te, y_te)
    print(f'{kernel:<8}  best C={best_C:.4f}  CV={max(cv_acc):.4f}  test={test_acc:.4f}')

---
## 3. Feature Expansion & the Kernel Trick

### When linear fails

Sometimes no linear boundary works regardless of C — the classes form concentric
circles or interleaving spirals. The solution is to **enlarge the feature space**.

In [ ]:
# Linear SVC fails on concentric circles
X_circ, y_circ = make_circles(n_samples=300, noise=0.1, factor=0.4, random_state=0)

svc_lin = Pipeline([('sc', StandardScaler()),
                    ('svc', SVC(kernel='linear', C=1))]).fit(X_circ, y_circ)

xx, yy = np.meshgrid(np.linspace(-1.5, 1.5, 300), np.linspace(-1.5, 1.5, 300))
Z = svc_lin.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

fig, ax = plt.subplots(figsize=(6, 5))
ax.contourf(xx, yy, Z, alpha=0.2, cmap='bwr')
ax.scatter(X_circ[y_circ==0,0], X_circ[y_circ==0,1], s=20)
ax.scatter(X_circ[y_circ==1,0], X_circ[y_circ==1,1], s=20, color='C3')
ax.set_title(f'Linear SVC on circles  (acc={svc_lin.score(X_circ,y_circ):.3f})')
plt.tight_layout(); plt.show()
print('A linear boundary cannot separate concentric circles — accuracy ≈ 50%.')

### Feature expansion: adding quadratic terms

Go from (X₁, X₂) to (X₁, X₂, X₁², X₂², X₁X₂). A linear SVC in this
5-D space produces a *conic section* boundary in the original 2-D space:

$$\beta_0 + \beta_1 X_1 + \beta_2 X_2 + \beta_3 X_1^2 + \beta_4 X_2^2 + \beta_5 X_1 X_2 = 0$$

This works but the dimensionality explodes for higher-degree polynomials —
$\binom{p+d}{d}$ basis functions for degree d and p predictors.

In [ ]:
# Manually expand features to degree 2, then fit linear SVC
def quad_expand(X):
    """Add X1^2, X2^2, X1*X2 to a 2-column array."""
    return np.c_[X, X[:,0]**2, X[:,1]**2, X[:,0]*X[:,1]]

X_exp = quad_expand(X_circ)
Xs    = StandardScaler().fit_transform(X_exp)
svc_exp = SVC(kernel='linear', C=1).fit(Xs, y_circ)

# Evaluate on a grid in original 2-D space
Xg = np.c_[xx.ravel(), yy.ravel()]
Xg_exp = StandardScaler().fit(X_exp).transform(quad_expand(Xg))
Z2 = svc_exp.predict(Xg_exp).reshape(xx.shape)

fig, ax = plt.subplots(figsize=(6, 5))
ax.contourf(xx, yy, Z2, alpha=0.2, cmap='bwr')
ax.scatter(X_circ[y_circ==0,0], X_circ[y_circ==0,1], s=20)
ax.scatter(X_circ[y_circ==1,0], X_circ[y_circ==1,1], s=20, color='C3')
ax.set_title(f'Linear SVC + quadratic features  (acc={svc_exp.score(Xs,y_circ):.3f})')
plt.tight_layout(); plt.show()
print(f'Expanded feature dimensions: {X_exp.shape[1]}  (was {X_circ.shape[1]})')

### Kernel lifting in 3-D

Adding the feature $r^2 = X_1^2 + X_2^2$ turns the concentric-circles problem into a
**linearly separable** one in 3-D. The separating plane in 3-D corresponds to a circular
boundary back in the original 2-D space.

In [ ]:
# 3-D view of the kernel trick: original circles become linearly separable
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401 (registers 3D projection)

r2 = X_circ[:, 0]**2 + X_circ[:, 1]**2   # radial feature

fig = plt.figure(figsize=(13, 5))

# Left: original 2-D space
ax1 = fig.add_subplot(121)
for cls, col, lbl in [(0, 'steelblue', 'inner'), (1, 'tomato', 'outer')]:
    mask = y_circ == cls
    ax1.scatter(X_circ[mask, 0], X_circ[mask, 1], c=col, s=20, alpha=0.7, label=lbl)
ax1.set_title('Original 2-D space — not linearly separable')
ax1.legend(); ax1.set_aspect('equal')

# Right: lifted 3-D space (X1, X2, r²)
ax2 = fig.add_subplot(122, projection='3d')
for cls, col in [(0, 'steelblue'), (1, 'tomato')]:
    mask = y_circ == cls
    ax2.scatter(X_circ[mask, 0], X_circ[mask, 1], r2[mask],
                c=col, s=15, alpha=0.7)
# Draw the separating plane  r² = threshold
threshold = (r2[y_circ == 0].max() + r2[y_circ == 1].min()) / 2
xg = np.linspace(-1.5, 1.5, 30)
xg1, xg2 = np.meshgrid(xg, xg)
ax2.plot_surface(xg1, xg2, np.full_like(xg1, threshold),
                 alpha=0.25, color='gold')
ax2.set_xlabel('X₁'); ax2.set_ylabel('X₂'); ax2.set_zlabel('X₁²+X₂²')
ax2.set_title('Lifted 3-D space — linearly separable\n(gold plane = decision boundary)')

plt.tight_layout()
plt.show()

### The kernel trick

The SVC classifier can be written using only **inner products** between training points:

$$f(x) = \beta_0 + \sum_{i \in \mathcal{S}} \hat{\alpha}_i \langle x, x_i \rangle$$

where $\mathcal{S}$ is the support set (indices with $\hat{\alpha}_i > 0$).

Replace the inner product with a **kernel function** $K(x_i, x_{i'})$ that implicitly
computes the inner product in a higher-dimensional space:

| Kernel | Formula | Properties |
|--------|---------|------------|
| Linear | $K(x_i, x_{i'}) = \sum_j x_{ij} x_{i'j}$ | Same as SVC |
| Polynomial | $K(x_i, x_{i'}) = \left(1 + \sum_j x_{ij} x_{i'j}\right)^d$ | $\binom{p+d}{d}$ implicit features |
| RBF | $K(x_i, x_{i'}) = \exp\left(-\gamma \sum_j (x_{ij} - x_{i'j})^2\right)$ | Infinite-dimensional; most flexible |

The RBF kernel decays with distance: K → 1 when points coincide, K → 0 when far apart.
γ controls the radius — small γ = wide kernel (smooth boundary), large γ = narrow (wiggly).

The kernel trick works because the dual SVM only ever needs inner products
$\langle \varphi(x_i), \varphi(x_j) \rangle$ — it never explicitly computes the feature
map φ. Replacing those inner products with $K(x_i, x_j)$ gives all the power of an
infinite-dimensional feature space at the cost of evaluating a simple scalar function.

In [ ]:
# Kernel comparison: linear / polynomial / RBF on make_moons
X_mn, y_mn = make_moons(n_samples=200, noise=0.25, random_state=0)

kernels = [
    # linear kernel SVC equivalent to support vector classifier — C is the only hyperparameter
    ('linear', {}),
    ('poly',   {'degree': 3}),
    # RBF kernel: K(x,z) = exp(-gamma*||x-z||^2) — maps to infinite-dimensional space implicitly
    ('rbf',    {'gamma': 'scale'}),
]

fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
for ax, (kernel, kw) in zip(axes, kernels):
    pipe = Pipeline([('sc', StandardScaler()),
                     ('svc', SVC(kernel=kernel, C=1, **kw))]).fit(X_mn, y_mn)
    xg = np.linspace(X_mn[:,0].min()-.5, X_mn[:,0].max()+.5, 300)
    yg = np.linspace(X_mn[:,1].min()-.5, X_mn[:,1].max()+.5, 300)
    xx2, yy2 = np.meshgrid(xg, yg)
    Z = pipe.predict(np.c_[xx2.ravel(), yy2.ravel()]).reshape(xx2.shape)
    ax.contourf(xx2, yy2, Z, alpha=0.2, cmap='bwr')
    ax.scatter(X_mn[y_mn==0,0], X_mn[y_mn==0,1], s=15)
    ax.scatter(X_mn[y_mn==1,0], X_mn[y_mn==1,1], s=15, color='C3')
    ax.set_title(f'{kernel}  (acc={pipe.score(X_mn,y_mn):.3f})', fontsize=11)
plt.suptitle('Kernel comparison on moons (C=1)', fontsize=12)
plt.tight_layout(); plt.show()

### Effect of γ in the RBF kernel

- **Small γ** → kernel is wide → distant points still influence each other → smooth, global boundary.
- **Large γ** → kernel is narrow → only nearby points influence each other → boundary wraps around individual training points → overfitting.

Intuitively, γ controls the "reach" of each training point: with large γ a training point
only influences predictions for inputs very close to it, so the boundary becomes highly
localised and can memorise every training example.

In [ ]:
# 4-panel: gamma effect on RBF SVM
# gamma = 1/(2*sigma^2): large gamma = tight kernel = complex boundary; small gamma = smooth boundary
gammas = [0.05, 0.5, 5.0, 50.0]
fig, axes = plt.subplots(2, 2, figsize=(11, 8))
for ax, g in zip(axes.flat, gammas):
    pipe = Pipeline([('sc', StandardScaler()),
                     ('svc', SVC(kernel='rbf', C=1, gamma=g))]).fit(X_mn, y_mn)
    xg = np.linspace(X_mn[:,0].min()-.5, X_mn[:,0].max()+.5, 300)
    yg = np.linspace(X_mn[:,1].min()-.5, X_mn[:,1].max()+.5, 300)
    xx2, yy2 = np.meshgrid(xg, yg)
    Z = pipe.predict(np.c_[xx2.ravel(), yy2.ravel()]).reshape(xx2.shape)
    ax.contourf(xx2, yy2, Z, alpha=0.2, cmap='bwr')
    ax.scatter(X_mn[y_mn==0,0], X_mn[y_mn==0,1], s=15)
    ax.scatter(X_mn[y_mn==1,0], X_mn[y_mn==1,1], s=15, color='C3')
    cv = cross_val_score(pipe, X_mn, y_mn, cv=5).mean()
    ax.set_title(f'γ={g}  |  train acc={pipe.score(X_mn,y_mn):.3f}  |  CV={cv:.3f}',
                 fontsize=10)
plt.suptitle('RBF kernel: effect of γ (C=1)', fontsize=12)
plt.tight_layout(); plt.show()
print('Large γ overfits: train accuracy rises but CV accuracy falls.')

### C × γ interaction — GridSearch heatmap

C and γ interact strongly. A low γ (wide kernel) needs a smaller C to avoid
overfitting; a high γ (tight kernel) already over-fits regardless of C.
The heatmap below makes the interaction visible.

In [ ]:
# Heatmap of 5-fold CV accuracy over a C × gamma grid
from sklearn.model_selection import GridSearchCV

C_vals_h   = [0.01, 0.1, 1, 10, 100]
gamma_vals = [0.001, 0.01, 0.1, 1, 10]

X_mn2, y_mn2 = make_moons(n_samples=300, noise=0.2, random_state=1)
grid_hm = GridSearchCV(
    Pipeline([('sc', StandardScaler()), ('svc', SVC(kernel='rbf'))]),
    {'svc__C': C_vals_h, 'svc__gamma': gamma_vals},
    cv=5, n_jobs=-1
).fit(X_mn2, y_mn2)

scores_mat = grid_hm.cv_results_['mean_test_score'].reshape(len(C_vals_h), len(gamma_vals))

fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(scores_mat, aspect='auto', cmap='YlGnBu', vmin=scores_mat.min())
ax.set_xticks(range(len(gamma_vals))); ax.set_xticklabels(gamma_vals)
ax.set_yticks(range(len(C_vals_h)));   ax.set_yticklabels(C_vals_h)
ax.set_xlabel('gamma'); ax.set_ylabel('C')
ax.set_title('5-fold CV accuracy — RBF SVM on make_moons')
plt.colorbar(im, ax=ax, label='CV accuracy')
for i in range(len(C_vals_h)):
    for j in range(len(gamma_vals)):
        ax.text(j, i, f'{scores_mat[i, j]:.2f}', ha='center', va='center',
                color='black' if scores_mat[i, j] > 0.85 else 'white', fontsize=9)
best_i, best_j = divmod(scores_mat.argmax(), len(gamma_vals))
ax.add_patch(plt.Rectangle((best_j - 0.5, best_i - 0.5), 1, 1,
                            fill=False, edgecolor='red', lw=2.5))
plt.tight_layout(); plt.show()
print(f'Best: C={grid_hm.best_params_["svc__C"]}, '
      f'gamma={grid_hm.best_params_["svc__gamma"]}, '
      f'CV={grid_hm.best_score_:.4f}')

---
## 4. ROC Curves, Multi-class SVMs & p ≫ n

### ROC curves

The SVC classifies by the sign of $\hat{f}(x)$. By varying a threshold t in
$\hat{f}(x) > t$ we sweep out a curve in (FPR, TPR) space — the **ROC curve**.
The **area under the curve (AUC)** summarises performance:

- AUC = 1.0 → perfect
- AUC = 0.5 → random guessing

sklearn's `SVC(probability=True)` enables probability estimates; `decision_function`
gives raw scores usable for ROC directly.

In [ ]:
# ROC curves: SVC vs LDA vs SVM with different gamma (Heart-style synthetic data)
# Generate a binary classification dataset (heart disease analogue)
from sklearn.datasets import make_classification
X_hrt, y_hrt = make_classification(n_samples=300, n_features=10, n_informative=6,
                                    n_redundant=2, random_state=42)
X_htr, X_hte, y_htr, y_hte = train_test_split(X_hrt, y_hrt, test_size=0.3, random_state=0)

scaler = StandardScaler().fit(X_htr)
Xtr_s  = scaler.transform(X_htr)
Xte_s  = scaler.transform(X_hte)

models = {
    'SVC (linear)': SVC(kernel='linear', C=1, probability=True).fit(Xtr_s, y_htr),
    'LDA'         : LinearDiscriminantAnalysis().fit(Xtr_s, y_htr),
    'SVM γ=0.001' : SVC(kernel='rbf', gamma=0.001, C=1, probability=True).fit(Xtr_s, y_htr),
    'SVM γ=0.01'  : SVC(kernel='rbf', gamma=0.01,  C=1, probability=True).fit(Xtr_s, y_htr),
    'SVM γ=0.1'   : SVC(kernel='rbf', gamma=0.1,   C=1, probability=True).fit(Xtr_s, y_htr),
}

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, split_label, Xtest, ytest in [
        (axes[0], 'Training set', Xtr_s, y_htr),
        (axes[1], 'Test set',     Xte_s, y_hte)]:
    for name, model in models.items():
        # decision_function returns signed distance to hyperplane — positive = class +1
        scores = model.predict_proba(Xtest)[:,1]
        fpr, tpr, _ = roc_curve(ytest, scores)
        ax.plot(fpr, tpr, lw=1.5, label=f'{name} (AUC={auc(fpr,tpr):.3f})')
    ax.plot([0,1],[0,1],'k--', lw=0.8)
    ax.set(xlabel='False positive rate', ylabel='True positive rate',
           title=f'ROC — {split_label}')
    ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

### Multi-class SVMs

SVMs are natively binary. Two extensions handle K > 2 classes:

**OVA (One-vs-All)**  
Fit K binary SVMs, each pitting class k against all others. Assign x* to the class
with the highest score $\hat{f}_k(x^*)$. K models, each sees all n samples.

**OVO (One-vs-One)**  
Fit $\binom{K}{2}$ binary SVMs, one for each pair. Assign x* by majority vote.
More classifiers but each only sees a subset of the data.

**Recommendation**: OVO when K is small — sklearn's SVC uses OVO by default
(`decision_function_shape='ovr'` switches to OVA-style scoring).

In practice OVO and OVA give very similar test accuracy; the choice rarely matters
as long as C and γ are tuned. sklearn's SVC uses OVO by default because each
sub-classifier trains on a smaller, balanced subset, which tends to be slightly faster.

In [ ]:
# Multi-class SVM on Iris (3 classes) — OVO (sklearn default)
iris = load_iris()
Xi, yi = iris.data[:, :2], iris.target   # use 2 features for visualisation

pipe_iris = Pipeline([('sc', StandardScaler()),
                      ('svc', SVC(kernel='rbf', C=5, gamma='scale'))]).fit(Xi, yi)

xx_i, yy_i = np.meshgrid(
    np.linspace(Xi[:,0].min()-0.5, Xi[:,0].max()+0.5, 300),
    np.linspace(Xi[:,1].min()-0.5, Xi[:,1].max()+0.5, 300))
Zi = pipe_iris.predict(np.c_[xx_i.ravel(), yy_i.ravel()]).reshape(xx_i.shape)

fig, ax = plt.subplots(figsize=(7, 5))
ax.contourf(xx_i, yy_i, Zi, alpha=0.25, cmap='tab10')
for cls, name in enumerate(iris.target_names):
    ax.scatter(Xi[yi==cls,0], Xi[yi==cls,1], s=30, label=name)
ax.set(xlabel=iris.feature_names[0], ylabel=iris.feature_names[1],
       title='Multi-class SVM (OVO, RBF kernel) on Iris')
ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

print(f'Number of OVO classifiers: {pipe_iris.named_steps["svc"].n_support_.shape[0]*(pipe_iris.named_steps["svc"].n_support_.shape[0]-1)//2}')
print(f'Train accuracy: {pipe_iris.score(Xi,yi):.4f}')
print('\nConfusion matrix:')
print(confusion_matrix(yi, pipe_iris.predict(Xi)))

### Decision-function confidence on Iris (OVO)

The SVC assigns each point to the class with the **highest pairwise vote count**.
Plotting the maximum decision score reveals high-confidence regions (solid colour)
and the uncertain border zone where classes contest.

In [ ]:
# Decision-function confidence overlay on the Iris 2-feature boundary
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Hard decision regions
ax = axes[0]
Z_hard = pipe_iris.predict(np.c_[xx_i.ravel(), yy_i.ravel()]).reshape(xx_i.shape)
ax.contourf(xx_i, yy_i, Z_hard, alpha=0.3, cmap='Set1')
scatter = ax.scatter(Xi[:, 0], Xi[:, 1], c=yi, cmap='Set1', edgecolors='k', s=25)
ax.set_title('Hard decision regions')
ax.set_xlabel('sepal length'); ax.set_ylabel('sepal width')

# Confidence = max decision score across classes
ax2 = axes[1]
df_vals = pipe_iris.decision_function(np.c_[xx_i.ravel(), yy_i.ravel()])
confidence = df_vals.max(axis=1).reshape(xx_i.shape)
cf = ax2.contourf(xx_i, yy_i, confidence, levels=20, cmap='plasma')
ax2.contour(xx_i, yy_i, Z_hard, colors='white', linewidths=0.8, alpha=0.6)
ax2.scatter(Xi[:, 0], Xi[:, 1], c=yi, cmap='Set1', edgecolors='k', s=25)
plt.colorbar(cf, ax=ax2, label='max decision score (confidence)')
ax2.set_title('Decision-function confidence\n(bright = high confidence)')
ax2.set_xlabel('sepal length'); ax2.set_ylabel('sepal width')

plt.tight_layout(); plt.show()

### Khan — p ≫ n (gene-expression data)

The Khan dataset is a classic **p ≫ n** problem:
- 63 training cases, 2308 gene-expression features
- 4 cancer subtypes (small round blue cell tumours)

Unregularised logistic regression fails — infinitely many solutions when p > n.
The SVM's margin objective implicitly regularises by finding the *minimum-norm*
β among all solutions, so a linear SVC works well without explicit regularisation.

In [ ]:
xtr = load_dataset('khan-xtrain').to_numpy()
ytr = load_dataset('khan-ytrain')['x'].to_numpy()
xte = load_dataset('khan-xtest').to_numpy()
yte = load_dataset('khan-ytest')['x'].to_numpy()
print(f'Train: {xtr.shape}  →  {np.bincount(ytr)[1:]} samples per class')
print(f'Test : {xte.shape}')

In [ ]:
# Fit linear SVC with C=10 (default from ISLR)
svc_khan = SVC(kernel='linear', C=10).fit(xtr, ytr)
print(f'Train accuracy = {svc_khan.score(xtr, ytr):.4f}')
print(f'Test  accuracy = {svc_khan.score(xte, yte):.4f}')
print('\nConfusion matrix (test):')
print(confusion_matrix(yte, svc_khan.predict(xte)))
print('\nClassification report (test):')
print(classification_report(yte, svc_khan.predict(xte)))

In [ ]:
# C and gamma interact strongly — always tune them jointly, not separately
grid_khan = GridSearchCV(
    SVC(kernel='linear'),
    {'C': [0.1, 1, 10, 100]},
    cv=3, n_jobs=-1
).fit(xtr, ytr)

print(f'Best C   : {grid_khan.best_params_["C"]}')
print(f'CV acc   : {grid_khan.best_score_:.4f}')
print(f'Test acc : {grid_khan.best_estimator_.score(xte, yte):.4f}')

---
## 5. SVM vs Logistic Regression

### SVM as loss + penalty

The SVC objective can be rewritten in the standard *loss + regularisation* form:

$$\min_{\beta_0,\boldsymbol{\beta}} \left\{ \sum_{i=1}^n \max\left[0,\; 1 - y_i f(x_i)\right] + \lambda \|\boldsymbol{\beta}\|^2 \right\}$$

The loss $\max[0, 1 - y_i f(x_i)]$ is the **hinge loss**:
- Zero when the prediction is correct and beyond the margin ($y_i f(x_i) \geq 1$).
- Linear otherwise.

Logistic regression uses the **log-loss** $\log(1 + e^{-y_i f(x_i)})$, which is always
positive. Both add an L2 penalty on β. The only structural difference is the loss shape.

In [ ]:
# Plot hinge loss vs log-loss
margin = np.linspace(-3, 3, 400)
hinge   = np.maximum(0, 1 - margin)
logloss = np.log1p(np.exp(-margin))

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(margin, hinge,   lw=2.5, label='Hinge loss (SVM)')
ax.plot(margin, logloss, lw=2.5, ls='--', label='Log-loss (Logistic regression)')
ax.axvline(0, color='grey', lw=0.8, alpha=0.5)
ax.axvline(1, color='grey', lw=0.8, ls=':', alpha=0.7, label='margin boundary (yf=1)')
ax.fill_between(margin, hinge, alpha=0.08)
ax.set(xlabel=r'$y_i \cdot f(x_i)$  (margin)', ylabel='Loss',
       title='Hinge loss vs Log-loss', ylim=(-0.2, 4))
ax.legend(fontsize=11); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print('Key difference:')
print('  Hinge loss is EXACTLY 0 for correctly classified, confident points (yf≥1).')
print('  Log-loss is always positive — pulls coefficients toward 0 even for correct points.')
print(f'  At margin=2: hinge={max(0,1-2):.1f}   log={float(np.log1p(np.exp(-2))):.4f}')

### When to use SVM vs Logistic Regression

| Situation | Recommendation |
|-----------|---------------|
| Classes are (nearly) separable | SVM (or LDA) outperforms LR |
| Classes overlap substantially | LR with ridge ≈ SVM; similar performance |
| Need calibrated probabilities | Logistic regression |
| Non-linear boundary | Kernel SVM (RBF) |
| p ≫ n (genomics, text) | Linear SVM; tune C |
| Interpretable coefficients | Logistic regression |

In [ ]:
# Side-by-side comparison: LR vs linear SVC vs RBF SVM on the heart-style data
results = {}
for name, model in [
    ('LR (ridge)',   Pipeline([('sc', StandardScaler()), ('lr', LogisticRegression(C=1, max_iter=1000))])),
    ('SVC linear',  Pipeline([('sc', StandardScaler()), ('svc', SVC(kernel='linear', C=1))])),
    ('SVM RBF',     Pipeline([('sc', StandardScaler()), ('svc', SVC(kernel='rbf', C=1, gamma='scale'))])),
]:
    cv = cross_val_score(model, X_hrt, y_hrt, cv=5).fit if False else \
         cross_val_score(model, X_hrt, y_hrt, cv=5)
    results[name] = {'CV mean': cv.mean(), 'CV std': cv.std()}

print(f'{"Model":<18} {"CV Accuracy":>12} {"Std":>8}')
print('-' * 40)
for name, r in results.items():
    print(f'{name:<18} {r["CV mean"]:>12.4f} {r["CV std"]:>8.4f}')

---
## 6. Recap

1. **Hyperplanes separate.** In p-D a hyperplane is a (p−1)-D flat surface defined by
   β₀ + β·X = 0. The MMC finds the unique widest gap; support vectors define it.

2. **Soft margin adds robustness.** Slack variables ε allow margin violations;
   C (sklearn) controls the budget. Large C → tight margin → overfits.

3. **Kernels give non-linearity for free.** Replace inner products with K(xᵢ,xᵢ').
   Polynomial and RBF kernels work in huge implicit feature spaces.

4. **Multi-class: OVO (K choose 2 classifiers, majority vote).** sklearn default.

5. **SVM for p ≫ n.** The margin objective regularises implicitly. Linear kernel + CV
   over C is the standard recipe for genomics and text.

6. **SVM ≈ LR when classes overlap.** Choose LR when you need probabilities;
   choose SVM when classes nearly separate or p ≫ n.

---
# Exercises

Each exercise is followed by its solution cell.
Try to solve it yourself before looking at the solution.

---
## Exercise 1

**Task.** Build `Pipeline(StandardScaler → SVC(kernel='rbf'))` and fit it on the
`penguins` dataset (drop NaN rows) to predict `species` from the four numeric
measurements. Report 5-fold CV accuracy.

In [ ]:
# your code here

### Exercise 1 — Solution

In [ ]:
df = load_dataset('penguins').dropna()
X_pen = df[['bill_length_mm','bill_depth_mm','flipper_length_mm','body_mass_g']]
y_pen = df['species']

pipe_pen = Pipeline([('sc', StandardScaler()), ('svc', SVC(kernel='rbf'))])
scores = cross_val_score(pipe_pen, X_pen, y_pen, cv=5)
print(f'5-fold CV accuracy: {scores.mean():.4f} ± {scores.std():.4f}')

---
## Exercise 2

**Task.** Now fit an *unscaled* RBF SVM on the same data (remove StandardScaler).
What happens to CV accuracy? Explain why.

In [ ]:
# your code here

### Exercise 2 — Solution

In [ ]:
scores_unscaled = cross_val_score(SVC(kernel='rbf'), X_pen, y_pen, cv=5)
print(f'Unscaled CV accuracy: {scores_unscaled.mean():.4f} ± {scores_unscaled.std():.4f}')
print(f'Scaled   CV accuracy: {scores.mean():.4f} ± {scores.std():.4f}')
print()
print('Without scaling, body_mass_g (range ~2700–6300 g) dominates the RBF')
print('distance calculation. The bill and flipper measurements (0–250 mm) are')
print('squeezed into a tiny range by comparison, so the kernel effectively ignores them.')
print(X_pen.describe().loc[['min','max']].round(1))

---
## Exercise 3

**Task.** Tune `C ∈ [0.1, 1, 10]` and `gamma ∈ ['scale', 0.1, 1.0]`
with 5-fold `GridSearchCV` on a `Pipeline(StandardScaler → SVC(rbf))`.
Report the best parameters and CV accuracy.

In [ ]:
# your code here

### Exercise 3 — Solution

In [ ]:
grid_pen = GridSearchCV(
    Pipeline([('sc', StandardScaler()), ('svc', SVC(kernel='rbf'))]),
    {'svc__C': [0.1, 1, 10], 'svc__gamma': ['scale', 0.1, 1.0]},
    cv=5, n_jobs=-1
).fit(X_pen, y_pen)

print(f'Best params : {grid_pen.best_params_}')
print(f'Best CV acc : {grid_pen.best_score_:.4f}')

---
## Exercise 4

**Task.** Fit an SVM with OVA strategy and an SVM with OVO strategy (sklearn default)
on the full Iris dataset (all 4 features). Use RBF kernel with `C=10, gamma='scale'`.
Compare 5-fold CV accuracy and print the confusion matrices on the full dataset.

In [ ]:
# your code here

### Exercise 4 — Solution

In [ ]:
iris_full = load_iris()
Xi4, yi4  = iris_full.data, iris_full.target

for strategy, shape in [('OVO (default)', 'ovo'), ('OVA', 'ovr')]:
    pipe_iris4 = Pipeline([
        ('sc',  StandardScaler()),
        ('svc', SVC(kernel='rbf', C=10, gamma='scale',
                    decision_function_shape=shape))
    ])
    cv4 = cross_val_score(pipe_iris4, Xi4, yi4, cv=5)
    pipe_iris4.fit(Xi4, yi4)
    print(f'\n{strategy}  CV acc = {cv4.mean():.4f} ± {cv4.std():.4f}')
    print('Confusion matrix:')
    print(confusion_matrix(yi4, pipe_iris4.predict(Xi4)))

---
## Exercise 5 — Kernel comparison grid

**Task.** Create three 2D toy datasets:

- (a) **Linearly separable** — two Gaussian blobs far apart (`make_blobs` with well-separated centers).
- (b) **Concentric circles** — `sklearn.datasets.make_circles(noise=0.1)`.
- (c) **Interleaved moons** — `sklearn.datasets.make_moons(noise=0.1)`.

For each dataset, fit `SVC` with `kernel='linear'`, `kernel='poly'` (degree=3), and `kernel='rbf'`.
Plot a **3×3 grid** of decision boundaries (rows = datasets, columns = kernels).

Which kernel works best on each dataset?

In [ ]:
# your code here

### Exercise 5 — Solution

In [ ]:
from sklearn.datasets import make_blobs, make_circles, make_moons

# (a) linearly separable blobs
X_a, y_a = make_blobs(n_samples=200, centers=[[-3, -3], [3, 3]], cluster_std=0.8, random_state=0)
# (b) concentric circles
X_b, y_b = make_circles(n_samples=200, noise=0.1, factor=0.4, random_state=0)
# (c) interleaved moons
X_c, y_c = make_moons(n_samples=200, noise=0.1, random_state=0)

datasets   = [('Blobs (linear)', X_a, y_a),
               ('Circles',        X_b, y_b),
               ('Moons',          X_c, y_c)]
# linear kernel SVC equivalent to support vector classifier — C is the only hyperparameter
kernel_configs = [
    ('linear', {}),
    ('poly',   {'degree': 3}),
    # RBF kernel: K(x,z) = exp(-gamma*||x-z||^2) — maps to infinite-dimensional space implicitly
    ('rbf',    {'gamma': 'scale'}),
]

fig, axes = plt.subplots(3, 3, figsize=(13, 11))

for row, (ds_name, X_ds, y_ds) in enumerate(datasets):
    xg = np.linspace(X_ds[:,0].min() - .5, X_ds[:,0].max() + .5, 300)
    yg = np.linspace(X_ds[:,1].min() - .5, X_ds[:,1].max() + .5, 300)
    xx_g, yy_g = np.meshgrid(xg, yg)

    for col, (kernel, kw) in enumerate(kernel_configs):
        ax = axes[row, col]
        # C controls the margin width: small C = wide margin (more violations allowed), large C = narrow margin (few violations)
        pipe = Pipeline([('sc', StandardScaler()),
                         ('svc', SVC(kernel=kernel, C=1, **kw))]).fit(X_ds, y_ds)
        Z = pipe.predict(np.c_[xx_g.ravel(), yy_g.ravel()]).reshape(xx_g.shape)
        ax.contourf(xx_g, yy_g, Z, alpha=0.25, cmap='bwr')
        ax.scatter(X_ds[y_ds==0, 0], X_ds[y_ds==0, 1], s=15)
        ax.scatter(X_ds[y_ds==1, 0], X_ds[y_ds==1, 1], s=15, color='C3')
        acc = pipe.score(X_ds, y_ds)
        ax.set_title(f'{ds_name} | {kernel}\nacc={acc:.3f}', fontsize=9)
        ax.set_xticks([]); ax.set_yticks([])

plt.suptitle('Kernel comparison grid (3 datasets × 3 kernels)', fontsize=12)
plt.tight_layout(); plt.show()

print('Linear wins on (a) — blobs are separable by a straight line.')
print('RBF wins on (b) and (c) — concentric/moon shapes need a non-linear boundary.')
print('Poly (degree=3) also handles (b)/(c) but is less robust to noise than RBF.')

---
## Exercise 6 — Support vector count vs C

**Task.** Using the synthetic Heart-style dataset (`X_hrt`, `y_hrt` from Section 4),
standardise the features and fit `SVC(kernel='rbf', gamma='scale')` for each
`C ∈ [0.001, 0.01, 0.1, 1, 10, 100, 1000]`.

For each value of C record:
- The **number of support vectors** (`len(svc.support_vectors_)`).
- The **test accuracy** on a held-out 30% split.

Plot both metrics on a **dual y-axis** chart with C on a log-scale x-axis.

What is the tradeoff between support vector count and accuracy as C increases?

In [ ]:
# your code here

### Exercise 6 — Solution

In [ ]:
from sklearn.model_selection import train_test_split

X_h6_tr, X_h6_te, y_h6_tr, y_h6_te = train_test_split(
    X_hrt, y_hrt, test_size=0.3, random_state=7)

sc6 = StandardScaler().fit(X_h6_tr)
Xtr6 = sc6.transform(X_h6_tr)
Xte6 = sc6.transform(X_h6_te)

# C controls the margin width: small C = wide margin (more violations allowed), large C = narrow margin (few violations)
C_list = [0.001, 0.01, 0.1, 1, 10, 100, 1000]
n_svs, test_accs = [], []

for C in C_list:
    svc6 = SVC(kernel='rbf', gamma='scale', C=C).fit(Xtr6, y_h6_tr)
    # only support vectors determine the boundary — the rest of the training data is irrelevant
    n_svs.append(len(svc6.support_vectors_))
    test_accs.append(svc6.score(Xte6, y_h6_te))

fig, ax1 = plt.subplots(figsize=(9, 5))
ax2 = ax1.twinx()

ax1.semilogx(C_list, n_svs,    'b-o', lw=2, label='# support vectors')
ax2.semilogx(C_list, test_accs,'r--s', lw=2, label='Test accuracy')

ax1.set_xlabel('C (log scale)')
ax1.set_ylabel('Number of support vectors', color='b')
ax2.set_ylabel('Test accuracy', color='r')
ax1.tick_params(axis='y', labelcolor='b')
ax2.tick_params(axis='y', labelcolor='r')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='center right')
ax1.set_title('Support vector count vs test accuracy as C varies (RBF, gamma=scale)')
ax1.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print('C_list       :', C_list)
print('# SVs        :', n_svs)
print('Test accuracy:', [f"{a:.3f}" for a in test_accs])
print()
print('As C increases: fewer support vectors (narrower margin), accuracy rises then plateaus.')
print('Very small C = underfitting (too many SVs, wide boundary ignores structure).')
print('Very large C = overfitting risk (near-zero margin, boundary hugs training points).')

---
## Exercise 7 — Dual coefficients and margin geometry

**Task.** Fit a linear SVC (`kernel='linear'`, `C=1.0`) on a **2D subset** of the
Heart-style dataset (use only the first two features after standardisation).

1. Identify the support vectors from `svc.support_vectors_`.
2. Plot:
   - All training points coloured by class.
   - Support vectors circled with a distinct marker.
   - The decision boundary (`w·x + b = 0`) and the two margin lines (`w·x + b = ±1`).
3. Print the **margin width** = $2 / \|\mathbf{w}\|$.

*Hint*: after fitting, the weight vector is `svc.coef_[0]` and the bias is `svc.intercept_[0]`.

In [ ]:
# your code here

### Exercise 7 — Solution

In [ ]:
# Use first two features of the heart-style dataset for a 2-D visualisation
X_2d = StandardScaler().fit_transform(X_hrt[:, :2])
y_2d = y_hrt

# linear kernel SVC equivalent to support vector classifier — C is the only hyperparameter
svc7 = SVC(kernel='linear', C=1.0).fit(X_2d, y_2d)

w7 = svc7.coef_[0]        # weight vector
b7 = svc7.intercept_[0]   # bias

# decision_function returns signed distance to hyperplane — positive = class +1
scores_7 = svc7.decision_function(X_2d)

# Compute decision boundary line: w·x + b = 0  →  x2 = -(w[0]*x1 + b) / w[1]
x1_range = np.linspace(X_2d[:,0].min() - 0.3, X_2d[:,0].max() + 0.3, 300)
db  = -(w7[0] * x1_range + b7)         / w7[1]   # decision boundary
m_up = -(w7[0] * x1_range + b7 - 1)   / w7[1]   # margin +1
m_dn = -(w7[0] * x1_range + b7 + 1)   / w7[1]   # margin -1

fig, ax = plt.subplots(figsize=(8, 6))
colors = ['steelblue', 'C3']
for cls in [0, 1]:
    mask = y_2d == cls
    ax.scatter(X_2d[mask, 0], X_2d[mask, 1], s=25, color=colors[cls],
               alpha=0.6, label=f'class {cls}')

# only support vectors determine the boundary — the rest of the training data is irrelevant
ax.scatter(svc7.support_vectors_[:, 0], svc7.support_vectors_[:, 1],
           s=140, facecolors='none', edgecolors='k', linewidths=1.8, label='support vectors')

ax.plot(x1_range, db,   'k-',  lw=2,   label='decision boundary  (w·x+b=0)')
ax.plot(x1_range, m_up, 'k--', lw=1.4, alpha=0.6, label='margin  (w·x+b=+1)')
ax.plot(x1_range, m_dn, 'k--', lw=1.4, alpha=0.6, label='margin  (w·x+b=−1)')

ax.set(xlabel='Feature 1 (standardised)', ylabel='Feature 2 (standardised)',
       title='Linear SVC — decision boundary & margin (2-D subset)',
       ylim=(X_2d[:,1].min() - 0.5, X_2d[:,1].max() + 0.5))
ax.legend(fontsize=9); ax.grid(alpha=0.2)
plt.tight_layout(); plt.show()

margin_width = 2 / np.linalg.norm(w7)
print(f'Weight vector w     : {w7}')
print(f'Bias b              : {b7:.4f}')
print(f'||w||               : {np.linalg.norm(w7):.4f}')
print(f'Margin width 2/||w||: {margin_width:.4f}')
# only support vectors determine the boundary — the rest of the training data is irrelevant
print(f'Number of SVs       : {len(svc7.support_vectors_)}')